# Часть 1 Бустинг (5 баллов)

В этой части будем предсказывать зарплату data scientist-ов в зависимости  от ряда факторов с помощью градиентного бустинга.

В датасете есть следующие признаки:



* work_year: The number of years of work experience in the field of data science.

* experience_level: The level of experience, such as Junior, Senior, or Lead.

* employment_type: The type of employment, such as Full-time or Contract.

* job_title: The specific job title or role, such as Data Analyst or Data Scientist.

* salary: The salary amount for the given job.

* salary_currency: The currency in which the salary is denoted.

* salary_in_usd: The equivalent salary amount converted to US dollars (USD) for comparison purposes.

* employee_residence: The country or region where the employee resides.

* remote_ratio: The percentage of remote work offered in the job.

* company_location: The location of the company or organization.

* company_size: The company's size is categorized as Small, Medium, or Large.

In [1]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
df = pd.read_csv("ds_salaries.csv")
df.head()

Saving ds_salaries.csv to ds_salaries (1).csv


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2023,SE,FT,Principal Data Scientist,80000,EUR,85847,ES,100,ES,L
1,2023,MI,CT,ML Engineer,30000,USD,30000,US,100,US,S
2,2023,MI,CT,ML Engineer,25500,USD,25500,US,100,US,S
3,2023,SE,FT,Data Scientist,175000,USD,175000,CA,100,CA,M
4,2023,SE,FT,Data Scientist,120000,USD,120000,CA,100,CA,M


## Задание 1 (0.5 балла) Подготовка



*   Разделите выборку на train, val, test (80%, 10%, 10%)
*   Выдерите salary_in_usd в качестве таргета
*   Найдите и удалите признак, из-за которого возможен лик в данных


In [ ]:
from sklearn.model_selection import train_test_split

y = df["salary_in_usd"]
X = df.drop(["salary", "salary_currency", "salary_in_usd"], axis=1)

X_ftrain, X_test, y_ftrain, y_test = train_test_split(X, y, train_size=0.9)
X_train, X_val, y_train, y_val = train_test_split(X_ftrain, y_ftrain, train_size=0.888)#<- 0.8/0.9

## Задание 2 (0.5 балла) Линейная модель


*   Закодируйте категориальные  признаки с помощью OneHotEncoder
*   Обучите модель линейной регрессии
*   Оцените  качество через MAPE и RMSE


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder
import numpy as np


categorical_features = ["experience_level", "employment_type", "job_title", "employee_residence", "company_location", "company_size"]
numeric_features = ['work_year', 'remote_ratio']

val_encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
full_encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)


X_train_encoded = val_encoder.fit_transform(X_train[categorical_features])
X_val_encoded = val_encoder.transform(X_val[categorical_features])

X_train_final = np.hstack([X_train[numeric_features].values, X_train_encoded])
X_val_final = np.hstack([X_val[numeric_features].values, X_val_encoded])


X_ftrain_encoded = full_encoder.fit_transform(X_ftrain[categorical_features])
X_test_encoded = full_encoder.transform(X_test[categorical_features])

X_ftrain_final = np.hstack([X_ftrain[numeric_features].values, X_ftrain_encoded])
X_test_final = np.hstack([X_test[numeric_features].values, X_test_encoded])


lr = LinearRegression()
lr.fit(X_ftrain_final, y_ftrain)

y_pred_ftrain = lr.predict(X_ftrain_final)
y_pred_test = lr.predict(X_test_final)


print(f"MAPE_ftrain: {mean_absolute_percentage_error(y_ftrain, y_pred_ftrain):.4f}")
print(f"RMSE_ftrain: {np.sqrt(mean_squared_error(y_ftrain, y_pred_ftrain)):.2f}")

print(f"MAPE_test: {mean_absolute_percentage_error(y_test, y_pred_test):.4f}")
print(f"RMSE_test: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.2f}")

MAPE_ftrain: 0.3076
RMSE_ftrain: 45490.52
MAPE_test: 0.3593
RMSE_test: 53080.28


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2, 3, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [3, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


## Задание 3 (0.5 балла) XGboost

Начнем с библиотеки xgboost.

Обучите модель `XGBRegressor` на тех же данных, что линейную модель, подобрав оптимальные гиперпараметры (`max_depth, learning_rate, n_estimators, gamma`, etc.) по валидационной выборке. Оцените качество итоговой модели (MAPE, RMSE), скорость обучения и скорость предсказания.

In [4]:
!pip install xgboost

import time
from xgboost import XGBRegressor

best_mape = float('inf')
best_params = {}

for md in [3, 5, 7, 9, 12]:
    for lr in [0.01, 0.03, 0.05, 0.1]:
        for ne in [100, 200, 300]:
            for gm in [0, 0.1, 0.5, 1]:
                model = XGBRegressor(max_depth=md, learning_rate=lr,
                                     n_estimators=ne, gamma=gm,
                                     random_state=42, verbosity=0)
                model.fit(X_train_final, y_train)
                pred_val = model.predict(X_val_final)
                mape = mean_absolute_percentage_error(y_val, pred_val)
                if mape < best_mape:
                    best_mape = mape
                    best_params = {'max_depth': md, 'learning_rate': lr,
                                   'n_estimators': ne, 'gamma': gm}

print(f"Лучшие параметры: {best_params}, MAPE на валидации: {best_mape:.4f}")

start_train = time.time()
final_model = XGBRegressor(**best_params, random_state=42)
final_model.fit(X_ftrain_final, y_ftrain)
train_time = time.time() - start_train

y_ftrain_pred = final_model.predict(X_ftrain_final)

start_pred = time.time()
y_test_pred = final_model.predict(X_test_final)
pred_time = time.time() - start_pred


print(f"MAPE_ftrain: {mean_absolute_percentage_error(y_ftrain, y_ftrain_pred):.4f}")
print(f"RMSE_ftrain: {np.sqrt(mean_squared_error(y_ftrain, y_ftrain_pred)):.2f}")

print(f"MAPE_test: {mean_absolute_percentage_error(y_test, y_test_pred):.4f}")
print(f"RMSE_test: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.2f}")

print(f"Время обучения: {train_time:.2f} с")
print(f"Время предсказания: {pred_time:.4f} с")

Лучшие параметры: {'max_depth': 12, 'learning_rate': 0.05, 'n_estimators': 300, 'gamma': 0}, MAPE на валидации: 0.3531
MAPE_ftrain: 0.2483
RMSE_ftrain: 40476.53
MAPE_test: 0.3337
RMSE_test: 49970.91
Время обучения: 0.82 с
Время предсказания: 0.0063 с


## Задание 4 (1 балл) CatBoost

Теперь библиотека CatBoost.

Обучите модель `CatBoostRegressor`, подобрав оптимальные гиперпараметры (`depth, learning_rate, iterations`, etc.) по валидационной выборке. Оцените качество итоговой модели (MAPE, RMSE), скорость обучения и скорость предсказания.

In [5]:
!pip install catboost

from catboost import CatBoostRegressor

best_mape = float('inf')
best_params = {}

for depth in [4, 5, 7, 9, 12]:
    for lr in [0.01, 0.03, 0.05, 0.1]:
        for iters in [100, 200, 300]:
            model = CatBoostRegressor(depth=depth, learning_rate=lr,
                                      iterations=iters, random_seed=42,
                                      verbose=False)
            model.fit(X_train_final, y_train)
            pred_val = model.predict(X_val_final)
            mape = mean_absolute_percentage_error(y_val, pred_val)
            if mape < best_mape:
                best_mape = mape
                best_params = {'depth': depth, 'learning_rate': lr, 'iterations': iters}

print(f"Лучшие параметры: {best_params}, MAPE на валидации: {best_mape:.4f}")

start_train = time.time()
final_model = CatBoostRegressor(**best_params, random_seed=42, verbose=False)
final_model.fit(X_ftrain_final, y_ftrain)
train_time = time.time() - start_train

y_ftrain_pred = final_model.predict(X_ftrain_final)
start_pred = time.time()
y_test_pred = final_model.predict(X_test_final)
pred_time = time.time() - start_pred

print(f"MAPE_ftrain: {mean_absolute_percentage_error(y_ftrain, y_ftrain_pred):.4f}")
print(f"RMSE_ftrain: {np.sqrt(mean_squared_error(y_ftrain, y_ftrain_pred)):.2f}")

print(f"MAPE_test: {mean_absolute_percentage_error(y_test, y_test_pred):.4f}")
print(f"RMSE_test: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.2f}")

print(f"Время обучения: {train_time:.2f} с")
print(f"Время предсказания: {pred_time:.4f} с")

Лучшие параметры: {'depth': 7, 'learning_rate': 0.1, 'iterations': 300}, MAPE на валидации: 0.3596
MAPE_ftrain: 0.3016
RMSE_ftrain: 42850.57
MAPE_test: 0.3428
RMSE_test: 48796.92
Время обучения: 0.71 с
Время предсказания: 0.0028 с


Для применения catboost моделей не обязательно сначала кодировать категориальные признаки, модель может кодировать их сама. Обучите catboost с подбором оптимальных гиперпараметров снова, используя pool для передачи данных в модель с указанием какие признаки категориальные, а какие нет с помощью параметра cat_features. Оцените качество и время. Стало ли лучше?

In [6]:
from catboost import Pool, CatBoostRegressor

cat_features = ["experience_level", "employment_type", "job_title",
                "employee_residence", "company_location", "company_size"]

train_pool = Pool(data=X_ftrain, label=y_ftrain, cat_features=cat_features)
val_pool = Pool(data=X_val, label=y_val, cat_features=cat_features)
test_pool = Pool(data=X_test, label=y_test, cat_features=cat_features)

best_mape = float('inf')
best_params = {}

for depth in [4, 5, 7, 9, 12]:
    for lr in [0.01, 0.03, 0.05, 0.1]:
        for iters in [100, 200, 300]:
            model = CatBoostRegressor(depth=depth, learning_rate=lr,
                                      iterations=iters, random_seed=42,
                                      verbose=False)
            model.fit(train_pool, eval_set=val_pool, verbose=False)
            pred_val = model.predict(val_pool)
            mape = mean_absolute_percentage_error(y_val, pred_val)
            if mape < best_mape:
                best_mape = mape
                best_params = {'depth': depth, 'learning_rate': lr, 'iterations': iters}

print(f"Лучшие параметры (с cat_features): {best_params}, MAPE на валидации: {best_mape:.4f}")

start_train = time.time()
final_model = CatBoostRegressor(**best_params, random_seed=42, verbose=False)
final_model.fit(train_pool, eval_set=val_pool, verbose=False)
train_time = time.time() - start_train

y_ftrain_pred = final_model.predict(train_pool)
start_pred = time.time()
y_test_pred = final_model.predict(test_pool)
pred_time = time.time() - start_pred

print(f"MAPE_ftrain: {mean_absolute_percentage_error(y_ftrain, y_ftrain_pred):.4f}")
print(f"RMSE_ftrain: {np.sqrt(mean_squared_error(y_ftrain, y_ftrain_pred)):.2f}")

print(f"MAPE_test: {mean_absolute_percentage_error(y_test, y_test_pred):.4f}")
print(f"RMSE_test: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.2f}")

print(f"Время обучения: {train_time:.2f} с")
print(f"Время предсказания: {pred_time:.4f} с")

Лучшие параметры (с cat_features): {'depth': 9, 'learning_rate': 0.1, 'iterations': 300}, MAPE на валидации: 0.3318
MAPE_ftrain: 0.3220
RMSE_ftrain: 44640.42
MAPE_test: 0.3571
RMSE_test: 50830.36
Время обучения: 6.52 с
Время предсказания: 0.0017 с


**Ответ:** Обучение проходило дольше, качекчтво меньше. Это может быть связано с размером выборки.

## Задание 5 (0.5 балла) LightGBM

И наконец библиотека LightGBM - используйте `LGBMRegressor`, снова подберите гиперпараметры, оцените качество и скорость.


In [7]:
!pip install lightgbm
from lightgbm import LGBMRegressor

best_mape = float('inf')
best_params = {}

for md in [3, 5, 7, 9, 12]:
    for lr in [0.01, 0.03, 0.05, 0.1]:
        for ne in [100, 200, 300]:
            model = LGBMRegressor(max_depth=md, learning_rate=lr,
                                  n_estimators=ne, random_state=42,
                                  verbose=-1)
            model.fit(X_train_final, y_train)
            pred_val = model.predict(X_val_final)
            mape = mean_absolute_percentage_error(y_val, pred_val)
            if mape < best_mape:
                best_mape = mape
                best_params = {'max_depth': md, 'learning_rate': lr, 'n_estimators': ne}

print(f"Лучшие параметры: {best_params}, MAPE на валидации: {best_mape:.4f}")

start_train = time.time()
final_model = LGBMRegressor(**best_params, random_state=42, verbose=-1)
final_model.fit(X_ftrain_final, y_ftrain)
train_time = time.time() - start_train

y_ftrain_pred = final_model.predict(X_ftrain_final)
start_pred = time.time()
y_test_pred = final_model.predict(X_test_final)
pred_time = time.time() - start_pred

print(f"MAPE_ftrain: {mean_absolute_percentage_error(y_ftrain, y_ftrain_pred):.4f}")
print(f"RMSE_ftrain: {np.sqrt(mean_squared_error(y_ftrain, y_ftrain_pred)):.2f}")

print(f"MAPE_test: {mean_absolute_percentage_error(y_test, y_test_pred):.4f}")
print(f"RMSE_test: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.2f}")

print(f"Время обучения: {train_time:.2f} с")
print(f"Время предсказания: {pred_time:.4f} с")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Лучшие параметры: {'max_depth': 12, 'learning_rate': 0.05, 'n_estimators': 100}, MAPE на валидации: 0.3846
MAPE_ftrain: 0.3399
RMSE_ftrain: 45893.16
MAPE_test: 0.3508
RMSE_test: 50208.49
Время обучения: 0.07 с
Время предсказания: 0.0067 с


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


## Задание 6 (2 балла) Сравнение и выводы

Сравните модели бустинга и сделайте про них выводы, какая из моделей показала лучший/худший результат по качеству, скорости обучения и скорости предсказания? Как отличаются гиперпараметры для разных моделей?

**Ответ:** Модели показали примерно одинаковые результаты. Дольше всего обучалась catboost с pool - 6 сек, остальные - меньше секунды. Возможно разница была бы более заметна, если выборка была бы больше, или были бы выбраны другие параметы, но их поиск занимает много времени.  

# Часть 2 Кластеризация (5 баллов)

Будем работать с данными о том, каких исполнителей слушают пользователи музыкального сервиса.

Каждая строка таблицы - информация об одном пользователе. Каждый столбец - это исполнитель (The Beatles, Radiohead, etc.)

Для каждой пары (пользователь, исполнитель) в таблице стоит число - доля прослушивания этого исполнителя этим пользователем.


In [8]:
import pandas as pd
ratings = pd.read_excel("https://github.com/evgpat/edu_stepik_rec_sys/blob/main/datasets/sample_matrix.xlsx?raw=true", engine='openpyxl')
ratings.head()

,user,the beatles,radiohead,deathcab for cutie,coldplay,modest mouse,sufjan stevens,dylan. bob,red hot clili peppers,pink fluid,...,municipal waste,townes van zandt,curtis mayfield,jewel,lamb,michal w. smith,群星,agalloch,meshuggah,yellowcard
0,0,NaN,0.020417,NaN,NaN,NaN,NaN,NaN,0.030496,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,NaN,0.184962,0.024561,NaN,NaN,0.136341,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,NaN,NaN,0.028635,NaN,NaN,NaN,0.024559,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,0.043529,0.086281,0.034590,0.016712,0.015935,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Будем строить кластеризацию исполнителей: если двух исполнителей слушало много людей примерно одинаковую долю своего времени (то есть векторы близки в пространстве), то, возможно исполнители похожи. Эта информация может быть полезна при построении рекомендательных систем.

## Задание 1 (0.5 балла) Подготовка

Транспонируем матрицу ratings, чтобы по строкам стояли исполнители.

In [21]:
ratings = ratings.T

Выкиньте строку под названием `user`.

In [22]:
ratings = ratings.drop('user')

В таблице много пропусков, так как пользователи слушают не всех-всех исполнителей, чья музыка представлена в сервисе, а некоторое подмножество (обычно около 30 исполнителей)


Доля исполнителя в музыке, прослушанной  пользователем, равна 0, если пользователь никогда не слушал музыку данного музыканта, поэтому заполните пропуски нулями.



In [25]:
ratings = ratings.fillna(0)
ratings.sample()

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
james brown,0.0,0.0,0.0,0.068966,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Задание 2 (0.5 балла) Первая кластеризация

Примените KMeans с 5ю кластерами, сохраните полученные лейблы

In [28]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42)
labels = kmeans.fit_predict(ratings)

cluster_labels = pd.Series(labels, index=ratings.index)

cluster_sizes = cluster_labels.value_counts().sort_index()
print("Размеры кластеров:")
for cl, size in cluster_sizes.items():
    print(f"Кластер {cl}: {size} исполнителей")

Размеры кластеров:
Кластер 0: 115 исполнителей
Кластер 1: 1 исполнителей
Кластер 2: 1 исполнителей
Кластер 3: 882 исполнителей
Кластер 4: 1 исполнителей


Выведите размеры кластеров. Полезной ли получилась кластеризация? Почему KMeans может выдать такой результат?

**Ответ:** Кластеризация получилось не очень полезной, т.к. 3 из 5 кластеров состоят из одного исполнителя. Могу предположить, что это связано с аномальными данными этих исполнителей.

## Задание 3 (0.5 балла) Объяснение результатов

При кластеризации получилось $\geq 1$ кластера размера 1. Выведите исполнителей, которые составляют такие кластеры. Среди них должна быть группа The Beatles.

In [33]:
cluster_counts = cluster_labels.value_counts()
singleton_clusters = cluster_counts[cluster_counts == 1].index

print("Исполнители в кластерах размера 1:")
for cl in singleton_clusters:
    artist = cluster_labels[cluster_labels == cl].index[0]
    print(f"  Кластер {cl}: {artist}")

Исполнители в кластерах размера 1:
  Кластер 4: the beatles
  Кластер 2: niИ
  Кластер 1: 日dir en grey


Изучите данные, почему именно The Beatles выделяется?

Подсказка: посмотрите на долю пользователей, которые слушают каждого исполнителя, среднюю долю прослушивания.

In [34]:
print("\nСтатистика по этим исполнителям:")
for cl in singleton_clusters:
    artist = cluster_labels[cluster_labels == cl].index[0]
    listeners_share = (ratings.loc[artist] > 0).mean()
    mean_share = ratings.loc[artist].mean()
    print(f"\n{artist}:")
    print(f"  Доля слушающих пользователей: {listeners_share:.2%}")
    print(f"  Средняя доля прослушивания: {mean_share:.4f}")


Статистика по этим исполнителям:

the beatles:
  Доля слушающих пользователей: 33.42%
  Средняя доля прослушивания: 0.0184

niИ:
  Доля слушающих пользователей: 11.82%
  Средняя доля прослушивания: 0.0065

日dir en grey:
  Доля слушающих пользователей: 1.02%
  Средняя доля прослушивания: 0.0017


In [37]:
listener_share = (ratings > 0).mean(axis=1)
mean_share = ratings.mean(axis=1)

avg_listener = listener_share.mean()
avg_mean = mean_share.mean()

print("Средние значения по всем исполнителям")
print(f"Доля слушающих пользователей (в среднем): {avg_listener:.2%}")
print(f"Средняя доля прослушивания (в среднем): {avg_mean:.6f}")

Средние значения по всем исполнителям
Доля слушающих пользователей (в среднем): 2.68%
Средняя доля прослушивания (в среднем): 0.000993


**Ответ:** the beatles слушают примерно терть всех пользователей. Это значительно бельше других исполнителей.

## Задание 4 (0.5 балла) Улучшение кластеризации

Попытаемся избавиться от этой проблемы: нормализуйте данные при помощи `normalize`.

In [39]:
from sklearn.preprocessing import normalize

ratings_norm = normalize(ratings)
ratings_norm = pd.DataFrame(ratings_norm, index=ratings.index, columns=ratings.columns)

Примените KMeans с 5ю кластерами на преобразованной матрице, посмотрите на их размеры. Стало ли лучше? Может ли кластеризация быть полезной теперь?

In [40]:
kmeans_norm = KMeans(n_clusters=5, random_state=42)
labels_norm = kmeans_norm.fit_predict(ratings_norm)

cluster_labels_norm = pd.Series(labels_norm, index=ratings_norm.index)
print("Размеры кластеров после нормализации:")
for cl, size in cluster_labels_norm.value_counts().sort_index().items():
    print(f"Кластер {cl}: {size} исполнителей")

Размеры кластеров после нормализации:
Кластер 0: 79 исполнителей
Кластер 1: 162 исполнителей
Кластер 2: 133 исполнителей
Кластер 3: 140 исполнителей
Кластер 4: 486 исполнителей


**Ответ** # Теперь кластеризация имеет больше смысла. Выбиваются только 0-ой и 4-ый.

## Задание 5 (1 балл) Центроиды

Выведите для каждого кластера названия топ-10 исполнителей, ближайших к центроиду по косинусной мере. Проинтерпретируйте результат. Что можно сказать о смысле кластеров?

In [42]:
from scipy.spatial.distance import cosine


centroids = kmeans_norm.cluster_centers_

for cluster_id in range(5):
    distances = {}
    for idx, artist in enumerate(ratings_norm.index):
        vec = ratings_norm.iloc[idx].values
        dist = cosine(centroids[cluster_id], vec)
        distances[artist] = dist

    top10 = sorted(distances.items(), key=lambda x: x[1])[:10]

    print(f"\nКластер {cluster_id} (размер: {sum(labels_norm == cluster_id)} исполнителей):")
    print("Топ-10 исполнителей, ближайших к центроиду:")
    for rank, (artist, d) in enumerate(top10, 1):
        print(f"  {rank:2d}. {artist} (cosine distance = {d:.4f})")


Кластер 0 (размер: 79 исполнителей):
Топ-10 исполнителей, ближайших к центроиду:
   1. nas (cosine distance = 0.4639)
   2. jay-z (cosine distance = 0.4955)
   3. kanye west (cosine distance = 0.5201)
   4. lupe the gorilla (cosine distance = 0.5839)
   5. a tribe called quest (cosine distance = 0.5867)
   6. the roots featuring d'angelo (cosine distance = 0.5938)
   7. gangstarr (cosine distance = 0.6280)
   8. little brother (cosine distance = 0.6291)
   9. lil' wayne (cosine distance = 0.6326)
  10. murs and 9th wonder (cosine distance = 0.6352)

Кластер 1 (размер: 162 исполнителей):
Топ-10 исполнителей, ближайших к центроиду:
   1. fall out boy (cosine distance = 0.6349)
   2. the all-americian rejects (cosine distance = 0.6560)
   3. paramore (cosine distance = 0.6587)
   4. kelly clarkson (cosine distance = 0.6644)
   5. john mayer (cosine distance = 0.6750)
   6. the fray (cosine distance = 0.6752)
   7. maroon5 (cosine distance = 0.6883)
   8. dashboard confesssional (cosine d

**Ответ:** # -- YOUR ANSWER HERE --

## Задание 6 (1 балл) Визуализация

Хотелось бы как-то визуализировать полученную кластеризацию. Постройте точечные графики `plt.scatter` для нескольких пар признаков исполнителей, покрасив точки в цвета кластеров. Почему визуализации получились такими? Хорошо ли они отражают разделение на кластеры? Почему?

In [ ]:
import matplotlib.pyplot as plt

# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

Для визуализации данных высокой размерности существует метод t-SNE (стохастическое вложение соседей с t-распределением). Данный метод является нелинейным методом снижения размерности: каждый объект высокой размерности будет моделироваться объектов более низкой (например, 2) размерности таким образом, чтобы похожие объекты моделировались близкими, непохожие - далекими с большой вероятностью.

Примените `TSNE` из библиотеки `sklearn` и визуализируйте полученные объекты, покрасив их в цвета их кластеров

In [ ]:
from sklearn.manifold import TSNE

# -- YOUR CODE HERE --

## Задание 7 (1 балл) Подбор гиперпараметров

Подберите оптимальное количество кластеров (максимум 100 кластеров) с использованием индекса Силуэта. Зафиксируйте `random_state=42`

In [ ]:
from sklearn.metrics import silhouette_score

# -- YOUR CODE HERE --

Выведите исполнителей, ближайших с центроидам (аналогично заданию 5). Как соотносятся результаты? Остался ли смысл кластеров прежним? Расскажите про смысл 1-2 интересных кластеров, если он изменился и кластеров слишком много, чтобы рассказать про все.

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

Сделайте t-SNE визуализацию полученной кластеризации.

In [ ]:
# -- YOUR CODE HERE --

Если кластеров получилось слишком много и визуально цвета плохо отличаются, покрасьте только какой-нибудь интересный кластер из задания выше (`c = (labels == i)`). Хорошо ли этот кластер отражается в визуализации?

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --